In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/ppp-round-2/train.csv
/kaggle/input/competitions/ppp-round-2/test.csv
/kaggle/input/competitions/ppp-round-2/PI1M.csv
/kaggle/input/competitions/ppp-round-2/archive/sample_submission.csv
/kaggle/input/competitions/ppp-round-2/archive/base_line_model.ipynb
/kaggle/input/competitions/ppp-round-2/archive/train.csv
/kaggle/input/competitions/ppp-round-2/archive/test.csv


In [2]:
"""
Molecular Property Prediction Pipeline -- FURTHER TIME-OPTIMIZED (v3)
=======================================================================

Builds on v2 (Mordred off, FP_NBITS=1024, boosting-round clamping, hist
tree_method, LightGBM-based feature selection). New changes in this version,
in rough order of impact:

1. ELIMINATED THE REDUNDANT REFIT IN optimize_model(). v2 clamped
   n_estimators/iterations to the winning trial's best_iteration, but to get
   that number it re-fit the model from scratch with early stopping AFTER
   study.optimize() finished -- a full extra boosting fit per model per
   target type, on top of the n_trials fits already done. That number was
   already computed once, inside the winning trial itself, and then thrown
   away. Now every objective() call stores its own best_iteration via
   trial.set_user_attr(), and optimize_model() reads it back off
   study.best_trial.user_attrs afterward -- zero extra fits, same clamp
   quality. This removes one full early-stopped LightGBM/XGBoost/CatBoost fit
   per model per target type.

2. REAL INTRA-TRIAL PRUNING for LightGBM and XGBoost. The MedianPruner was
   configured in both prior versions but the objective functions never
   called trial.report()/should_prune(), so it was inert -- every trial ran
   to its own early-stopping point regardless of how bad it looked early on.
   Now a per-round callback reports the validation score to Optuna each
   boosting round; once a trial is clearly worse than the running median at
   the same round, it's pruned (raises optuna.TrialPruned) instead of
   continuing to train. CatBoost intra-trial pruning needs a different
   mechanism (its Python callback API doesn't expose per-round eval values
   the same way) and isn't wired up here to keep scope contained -- its
   existing early_stopping_rounds is still the main lever for it.

3. Trimmed default search/CV budgets (still overridable via Config):
     OPTUNA_TRIALS   10 -> 7   (pruning now does a lot of the work trials 8-10 did)
     CV_FOLDS         5 -> 3   (still gives a reasonably stable OOF estimate)
     N_SELECTED_FEATURES 2000 -> 1200
     FP_NBITS        1024 -> 512  (still 4 distinct 512-bit fingerprints + MACCS)
     EARLY_STOPPING_ROUNDS 50 -> 30
   These are real accuracy/time trade-offs, not free lunches -- see the
   comment above cfg.FP_NBITS if you want to dial specific ones back up.

4. ENABLE_MLP now defaults to False. Of the five candidate models, MLP
   contributes the least to the final Ridge blend in this kind of tabular/
   fingerprint setting most of the time, while being the slowest of the
   "cheap" models once fingerprint width x N_SELECTED_FEATURES gets large
   (dense hidden layers over thousands of mostly-binary inputs). Re-enable
   it (cfg.ENABLE_MLP = True) if your data shows it's pulling meaningful
   weight in the "Blending weights" log line.

5. RandomForest n_estimators trimmed 500 -> 250 in _build_model(). RF has no
   early-stopping mechanism, so unlike the boosters this is the only lever
   for it; 250 trees is still plenty for OOB-stable predictions on typical
   Kaggle-scale data, at half the wall time.

CARRIED FORWARD FROM THE PRIOR PATCH: HyperparameterOptimizer.optimize_model()
still merges random_state / n_jobs / thread_count (and, for lightgbm,
verbose) into the best_params it returns -- study.best_params only ever
contains keys that came from trial.suggest_*() calls, so without this merge
every downstream CV fold and the final fit would silently run without
cfg.RANDOM_SEED / cfg.N_CORES. CatBoost's 'verbose' is still deliberately
excluded since _build_model() passes verbose=False explicitly for it.
"""

import os
import gc
import logging
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple, Any

!pip install rdkit
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import RepeatedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

import lightgbm as lgb
import xgboost as xgb
try:
    import catboost as cb
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False

from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, MACCSkeys, AllChem
from rdkit.Avalon import pyAvalonTools

try:
    from mordred import Calculator, descriptors as mordred_descriptors
    MORDRED_AVAILABLE = True
except ImportError:
    MORDRED_AVAILABLE = False

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

warnings.filterwarnings('ignore')

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger('MolPropPrediction')
optuna.logging.set_verbosity(optuna.logging.WARNING)


@dataclass
class Config:
    RANDOM_SEED: int = 42
    N_CORES: int = max(1, os.cpu_count() or 2)
    USE_CACHING: bool = True
    CACHE_DIR: str = './cache'

    USE_RDKIT_DESCS: bool = True
    USE_MORDRED: bool = False
    USE_MORGAN: bool = True
    USE_ATOM_PAIR: bool = True
    USE_TORSION: bool = True
    USE_AVALON: bool = True
    USE_MACCS: bool = True
    USE_GRAPH_STATS: bool = True
    USE_FRAGMENTS: bool = False
    # TIME OPT (v3): 1024 -> 512. Still 4 distinct fingerprint families + MACCS;
    # bump back to 1024/2048 if you see feature-importance selection consistently
    # saturating N_SELECTED_FEATURES with fingerprint bits and suspect collisions.
    FP_NBITS: int = 512

    VARIANCE_THRESHOLD: float = 0.01
    CORRELATION_THRESHOLD: float = 0.95
    USE_PCA: bool = False
    PCA_VARIANCE: float = 0.95
    FEATURE_SELECTION_METHOD: Optional[str] = 'importance'
    N_SELECTED_FEATURES: int = 1200  # TIME OPT (v3): was 2000

    TARGET_TYPE_STRATEGY: str = 'separate_models'
    CV_FOLDS: int = 3  # TIME OPT (v3): was 5
    CV_REPEATS: int = 1
    SEED_AVERAGING_N: int = 1
    OPTUNA_TRIALS: int = 7  # TIME OPT (v3): was 10; pruning covers some of the gap
    EARLY_STOPPING_ROUNDS: int = 30  # TIME OPT (v3): was 50
    USE_PSEUDO_LABELING: bool = False

    ENABLE_LGBM: bool = True
    ENABLE_XGBOOST: bool = True
    ENABLE_CATBOOST: bool = CATBOOST_AVAILABLE
    ENABLE_RF: bool = True
    ENABLE_HGB: bool = True
    ENABLE_MLP: bool = False  # TIME OPT (v3): off by default -- see module docstring
    ENABLE_ELASTICNET: bool = False


cfg = Config()

_HAS_FAST_RDKIT_DESCS = hasattr(Descriptors, 'CalcMolDescriptors')


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    import random
    random.seed(seed)


class DataLoader:
    @staticmethod
    def find_files(input_dir: str) -> Tuple[str, str]:
        train_path, test_path = None, None
        for dirname, _, filenames in os.walk(input_dir):
            for f in filenames:
                if f.lower().endswith('.csv'):
                    if 'train' in f.lower() and train_path is None:
                        train_path = os.path.join(dirname, f)
                    elif 'test' in f.lower() and test_path is None:
                        test_path = os.path.join(dirname, f)
        if train_path is None or test_path is None:
            raise FileNotFoundError("Could not find train.csv or test.csv.")
        return train_path, test_path

    @staticmethod
    def load_data(train_path: str, test_path: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
        train_df = pd.read_csv(train_path).reset_index(drop=True)
        test_df = pd.read_csv(test_path).reset_index(drop=True)
        if 'id' not in test_df.columns:
            test_df['id'] = test_df.index
        train_df['target'] = pd.to_numeric(train_df['target'], errors='coerce').astype(float)
        train_df = train_df.dropna(subset=['target'])
        logger.info(f"Loaded {len(train_df)} train samples, {len(test_df)} test samples.")
        return train_df, test_df


class DescriptorGenerator:
    _rdkit_desc_names = [name for name, _ in Descriptors.descList]

    def __init__(self, n_jobs: int = 4, use_cache: bool = True):
        self.n_jobs = max(1, n_jobs)
        self.use_cache = use_cache
        Path(cfg.CACHE_DIR).mkdir(exist_ok=True)
        self.feature_names: Optional[List[str]] = None

    @staticmethod
    def _smiles_to_mol(smiles: str):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is not None:
                Chem.SanitizeMol(mol)
            return mol
        except Exception:
            return None

    @staticmethod
    def _fp_to_array(fp, nbits: int) -> np.ndarray:
        arr = np.zeros((nbits,), dtype=np.int8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        return arr

    @classmethod
    def _compute_row(cls, smiles: str) -> Optional[np.ndarray]:
        mol = cls._smiles_to_mol(smiles)
        if mol is None:
            return None

        chunks = []
        nbits = cfg.FP_NBITS

        if cfg.USE_RDKIT_DESCS:
            if _HAS_FAST_RDKIT_DESCS:
                try:
                    desc_dict = Descriptors.CalcMolDescriptors(mol)
                    vals = [desc_dict.get(name, 0.0) for name in cls._rdkit_desc_names]
                except Exception:
                    vals = [0.0] * len(cls._rdkit_desc_names)
            else:
                vals = []
                for _, func in Descriptors.descList:
                    try:
                        vals.append(func(mol))
                    except Exception:
                        vals.append(0.0)
            chunks.append(np.asarray(vals, dtype=np.float64))

        if cfg.USE_MACCS:
            chunks.append(cls._fp_to_array(MACCSkeys.GenMACCSKeys(mol), 167).astype(np.float64))

        if cfg.USE_MORGAN:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=nbits)
            chunks.append(cls._fp_to_array(fp, nbits).astype(np.float64))

        if cfg.USE_ATOM_PAIR:
            fp = AllChem.GetHashedAtomPairFingerprintAsBitVect(mol, nBits=nbits)
            chunks.append(cls._fp_to_array(fp, nbits).astype(np.float64))

        if cfg.USE_TORSION:
            fp = AllChem.GetHashedTopologicalTorsionFingerprintAsBitVect(mol, nBits=nbits)
            chunks.append(cls._fp_to_array(fp, nbits).astype(np.float64))

        if cfg.USE_AVALON:
            fp = pyAvalonTools.GetAvalonFP(mol, nBits=nbits)
            chunks.append(cls._fp_to_array(fp, nbits).astype(np.float64))

        if cfg.USE_GRAPH_STATS or cfg.USE_FRAGMENTS:
            atom_count = mol.GetNumAtoms()
            ring_info = mol.GetRingInfo()
            chunks.append(np.array([
                atom_count,
                mol.GetNumBonds(),
                sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() == 6) / max(atom_count, 1),
                sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() == 7) / max(atom_count, 1),
                sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() == 8) / max(atom_count, 1),
                ring_info.NumRings(),
                np.mean([len(r) for r in ring_info.AtomRings()]) if ring_info.NumRings() > 0 else 0.0,
            ], dtype=np.float64))

        return np.concatenate(chunks) if chunks else np.zeros((0,), dtype=np.float64)

    def _build_feature_names(self) -> List[str]:
        names: List[str] = []
        nbits = cfg.FP_NBITS
        if cfg.USE_RDKIT_DESCS:
            names += self._rdkit_desc_names
        if cfg.USE_MACCS:
            names += [f'MACCS_{i}' for i in range(167)]
        if cfg.USE_MORGAN:
            names += [f'Morgan_{i}' for i in range(nbits)]
        if cfg.USE_ATOM_PAIR:
            names += [f'AtomPair_{i}' for i in range(nbits)]
        if cfg.USE_TORSION:
            names += [f'Torsion_{i}' for i in range(nbits)]
        if cfg.USE_AVALON:
            names += [f'Avalon_{i}' for i in range(nbits)]
        if cfg.USE_GRAPH_STATS or cfg.USE_FRAGMENTS:
            names += ['Graph_NumAtoms', 'Graph_NumBonds', 'Graph_FractionC',
                      'Graph_FractionN', 'Graph_FractionO', 'Graph_RingCount', 'Graph_RingSizeAvg']
        return names

    def _append_mordred(self, df: pd.DataFrame, smiles_series: pd.Series) -> pd.DataFrame:
        logger.info("Computing Mordred descriptors in a single batch call...")
        calc = Calculator(mordred_descriptors, ignore_3D=True)
        mols = [self._smiles_to_mol(s) for s in smiles_series]
        mordred_df = calc.pandas(mols, nproc=self.n_jobs, quiet=True)
        mordred_df.columns = [f'Mordred_{c}' for c in mordred_df.columns]
        mordred_df.index = df.index
        mordred_df = mordred_df.apply(pd.to_numeric, errors='coerce').fillna(0.0)
        return pd.concat([df, mordred_df], axis=1)

    def generate(self, smiles_series: pd.Series, cache_name: str) -> pd.DataFrame:
        logger.info(f"Generating descriptors for {len(smiles_series)} molecules ({cache_name})...")
        cache_path = Path(cfg.CACHE_DIR) / f'{cache_name}.parquet'

        if self.use_cache and cache_path.exists():
            logger.info(f"Loading cached features from {cache_path}")
            return pd.read_parquet(cache_path)

        if self.feature_names is None:
            self.feature_names = self._build_feature_names()

        smiles_list = smiles_series.tolist()
        idx_list = smiles_series.index.tolist()
        chunksize = max(1, len(smiles_list) // (self.n_jobs * 4))

        rows, valid_idx = [], []
        from multiprocessing import Pool
        with Pool(processes=self.n_jobs) as pool:
            for idx, row in tqdm(
                zip(idx_list, pool.imap(self._compute_row, smiles_list, chunksize=chunksize)),
                total=len(smiles_list), desc=f"Descriptors[{cache_name}]"
            ):
                if row is not None and row.size > 0:
                    rows.append(row)
                    valid_idx.append(idx)

        X = np.vstack(rows) if rows else np.empty((0, len(self.feature_names)))
        df = pd.DataFrame(X, columns=self.feature_names, index=valid_idx)
        df = df.replace([np.inf, -np.inf], 0.0).fillna(0.0)
        logger.info(f"Generated {df.shape[1]} base features for {df.shape[0]} valid molecules.")

        if cfg.USE_MORDRED and MORDRED_AVAILABLE:
            df = self._append_mordred(df, smiles_series.loc[valid_idx])

        if self.use_cache:
            df.to_parquet(cache_path)
        return df


class FeaturePipeline:
    def __init__(self):
        self.scaler = None
        self.var_selector = None
        self.pca = None
        self.variance_cols: List[str] = []

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.replace([np.inf, -np.inf], 0.0)

        self.var_selector = VarianceThreshold(threshold=cfg.VARIANCE_THRESHOLD)
        X_np = self.var_selector.fit_transform(X)
        self.variance_cols = X.columns[self.var_selector.get_support()].tolist()
        logger.info(f"After VarianceThreshold: {len(self.variance_cols)} features left.")

        self.scaler = StandardScaler()
        X_scaled = self.scaler.fit_transform(X_np)

        if cfg.USE_PCA:
            self.pca = PCA(n_components=cfg.PCA_VARIANCE, random_state=cfg.RANDOM_SEED)
            X_scaled = self.pca.fit_transform(X_scaled)
            logger.info(f"PCA reduced to {X_scaled.shape[1]} components.")

        return pd.DataFrame(X_scaled, index=X.index)

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        # VarianceThreshold.transform() expects the SAME full column set seen at fit()
        # time (it applies the selection internally) -- do not pre-subset here.
        X = X.replace([np.inf, -np.inf], 0.0)
        X_np = self.var_selector.transform(X)
        X_scaled = self.scaler.transform(X_np)
        if cfg.USE_PCA:
            X_scaled = self.pca.transform(X_scaled)
        return pd.DataFrame(X_scaled, index=X.index)


# --- Pruning callbacks (new in v3) ---
def _make_lgb_pruning_callback(trial: optuna.trial.Trial):
    """Reports the latest validation score to Optuna after every boosting round
    and raises optuna.TrialPruned() once the trial is clearly underperforming
    relative to other trials at the same round. Makes the configured
    MedianPruner actually do something for LightGBM (previously inert)."""
    def _callback(env):
        if not env.evaluation_result_list:
            return
        current_score = env.evaluation_result_list[-1][2]
        trial.report(current_score, step=env.iteration)
        if trial.should_prune():
            raise optuna.TrialPruned(f"Pruned at LightGBM iteration {env.iteration}")
    _callback.order = 30
    return _callback


class _XGBPruningCallback(xgb.callback.TrainingCallback):
    """Same idea as the LightGBM callback above, for XGBoost's callback API."""
    def __init__(self, trial: optuna.trial.Trial):
        self.trial = trial

    def after_iteration(self, model, epoch, evals_log) -> bool:
        if not evals_log:
            return False
        last_eval = list(evals_log.values())[-1]
        last_metric = list(last_eval.values())[-1]
        if not last_metric:
            return False
        current_score = last_metric[-1]
        self.trial.report(current_score, step=epoch)
        if self.trial.should_prune():
            raise optuna.TrialPruned(f"Pruned at XGBoost iteration {epoch}")
        return False


class HyperparameterOptimizer:
    @staticmethod
    def suggest_params(trial: optuna.trial.Trial, model_name: str) -> dict:
        if model_name == 'lightgbm':
            return {
                'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
                'num_leaves': trial.suggest_int('num_leaves', 15, 255),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
                'verbose': -1, 'random_state': cfg.RANDOM_SEED, 'n_jobs': cfg.N_CORES
            }
        elif model_name == 'xgboost':
            return {
                'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
                'tree_method': 'hist',
                'random_state': cfg.RANDOM_SEED, 'n_jobs': cfg.N_CORES
            }
        elif model_name == 'catboost':
            return {
                'iterations': trial.suggest_int('iterations', 500, 1500),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'depth': trial.suggest_int('depth', 4, 10),
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
                'border_count': trial.suggest_int('border_count', 32, 255),
                'random_state': cfg.RANDOM_SEED, 'verbose': False, 'thread_count': cfg.N_CORES
            }
        return {}

    @staticmethod
    def optimize_model(model_name: str, X_train, y_train, X_val, y_val, n_trials: int = 15) -> dict:
        def objective(trial):
            params = HyperparameterOptimizer.suggest_params(trial, model_name)
            try:
                if model_name == 'lightgbm':
                    model = lgb.LGBMRegressor(**params)
                    model.fit(
                        X_train, y_train, eval_set=[(X_val, y_val)],
                        callbacks=[
                            lgb.early_stopping(cfg.EARLY_STOPPING_ROUNDS, verbose=False),
                            _make_lgb_pruning_callback(trial),
                        ]
                    )
                    best_iter = getattr(model, 'best_iteration_', None) or params.get('n_estimators', 500)
                elif model_name == 'xgboost':
                    params['eval_metric'] = 'rmse'
                    params['early_stopping_rounds'] = cfg.EARLY_STOPPING_ROUNDS
                    params['callbacks'] = [_XGBPruningCallback(trial)]
                    model = xgb.XGBRegressor(**params)
                    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
                    best_iter = getattr(model, 'best_iteration', None) or params.get('n_estimators', 500)
                elif model_name == 'catboost':
                    # No intra-trial pruning wired up for CatBoost -- see module docstring.
                    model = cb.CatBoostRegressor(**params)
                    model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                              early_stopping_rounds=cfg.EARLY_STOPPING_ROUNDS, verbose=False)
                    best_iter = model.get_best_iteration() or params.get('iterations', 500)
                else:
                    return float('inf')

                # PATCH (v3): stash best_iteration on the trial itself so optimize_model()
                # can read it back after study.optimize() without a redundant refit.
                trial.set_user_attr('best_iteration', int(best_iter))

                preds = model.predict(X_val)
                return np.sqrt(mean_squared_error(y_val, preds))
            except optuna.TrialPruned:
                raise
            except Exception as e:
                logger.warning(f"Trial failed: {e}")
                return float('inf')

        study = optuna.create_study(
            direction='minimize',
            sampler=TPESampler(seed=cfg.RANDOM_SEED),
            pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=20, interval_steps=5)
        )
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

        best_params = dict(study.best_params) if study.best_params else {}
        if not best_params:
            return best_params

        # --- Clamp boosting rounds using the best trial's own recorded iteration ---
        # No refit needed: objective() already stored it via set_user_attr above.
        best_iter = study.best_trial.user_attrs.get('best_iteration')
        default_key = 'iterations' if model_name == 'catboost' else 'n_estimators'
        if best_iter is None:
            best_iter = best_params.get(default_key, 500)
        best_params[default_key] = max(50, int(best_iter * 1.05))

        # Merge static (non-suggested) params back in -- study.best_params only ever
        # contains keys that came from trial.suggest_*() calls (see module docstring).
        if model_name == 'lightgbm':
            best_params['verbose'] = -1
            best_params['random_state'] = cfg.RANDOM_SEED
            best_params['n_jobs'] = cfg.N_CORES
        elif model_name == 'xgboost':
            best_params['tree_method'] = 'hist'
            best_params['random_state'] = cfg.RANDOM_SEED
            best_params['n_jobs'] = cfg.N_CORES
        elif model_name == 'catboost':
            best_params['random_state'] = cfg.RANDOM_SEED
            best_params['thread_count'] = cfg.N_CORES
            # 'verbose' intentionally excluded -- _build_model() sets it explicitly.

        return best_params


class ModelTrainer:
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.best_params: dict = {}
        self.models: Dict[str, Any] = {}
        self.oof_preds = None

    def _build_model(self, params: dict):
        if self.model_name == 'lightgbm':
            return lgb.LGBMRegressor(**params)
        elif self.model_name == 'xgboost':
            params = {'tree_method': 'hist', **params}
            return xgb.XGBRegressor(**params)
        elif self.model_name == 'catboost':
            return cb.CatBoostRegressor(**params, verbose=False)
        elif self.model_name == 'rf':
            # TIME OPT (v3): 500 -> 250 trees; RF has no early-stopping lever.
            return RandomForestRegressor(n_estimators=250, random_state=cfg.RANDOM_SEED, n_jobs=cfg.N_CORES)
        elif self.model_name == 'et':
            return ExtraTreesRegressor(n_estimators=250, random_state=cfg.RANDOM_SEED, n_jobs=cfg.N_CORES)
        elif self.model_name == 'mlp':
            return MLPRegressor(hidden_layer_sizes=(256, 128), max_iter=300, random_state=cfg.RANDOM_SEED,
                                 early_stopping=True, n_iter_no_change=10, validation_fraction=0.1)
        elif self.model_name == 'elasticnet':
            return ElasticNet(random_state=cfg.RANDOM_SEED, max_iter=1000)
        raise ValueError(f"Unknown model: {self.model_name}")

    def fit_cv(self, X: np.ndarray, y: np.ndarray, y_orig: Optional[np.ndarray] = None):
        logger.info(f"Training {self.model_name} with {cfg.CV_REPEATS}x{cfg.CV_FOLDS} fold CV...")

        if self.model_name in ('lightgbm', 'xgboost', 'catboost'):
            X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=cfg.RANDOM_SEED)
            self.best_params = HyperparameterOptimizer.optimize_model(
                self.model_name, X_tr, y_tr, X_val, y_val, n_trials=cfg.OPTUNA_TRIALS
            )

        seeds = [cfg.RANDOM_SEED + i for i in range(cfg.SEED_AVERAGING_N)]
        all_oof_preds = []

        n_bins = min(10, max(2, len(np.unique(y))))
        try:
            y_binned = pd.cut(y, bins=n_bins, labels=False, duplicates='drop')
        except Exception:
            y_binned = pd.Series(np.zeros_like(y), index=range(len(y)))

        kf = RepeatedKFold(n_splits=cfg.CV_FOLDS, n_repeats=cfg.CV_REPEATS, random_state=cfg.RANDOM_SEED)

        for seed in seeds:
            set_seed(seed)
            oof_preds = np.zeros(len(y))
            for train_idx, val_idx in tqdm(kf.split(X, y_binned), total=cfg.CV_FOLDS * cfg.CV_REPEATS,
                                            desc=f"{self.model_name} fold"):
                X_tr, X_val = X[train_idx], X[val_idx]
                y_tr, y_val = y[train_idx], y[val_idx]
                model = self._build_model(self.best_params)
                model.fit(X_tr, y_tr)
                oof_preds[val_idx] = model.predict(X_val)
            all_oof_preds.append(oof_preds)

        final_oof = np.mean(all_oof_preds, axis=0)
        self.oof_preds = final_oof

        rmse_log = np.sqrt(mean_squared_error(y, final_oof))
        r2_log = r2_score(y, final_oof)
        logger.info(f"{self.model_name} -> LogScale CV RMSE: {rmse_log:.6f}, CV R2: {r2_log:.6f}")

        if y_orig is not None:
            final_oof_orig = np.expm1(final_oof)
            y_orig_transformed = np.expm1(y_orig)
            rmse_orig = np.sqrt(mean_squared_error(y_orig_transformed, final_oof_orig))
            r2_orig = r2_score(y_orig_transformed, final_oof_orig)
            logger.info(f"{self.model_name} -> OrigScale CV RMSE: {rmse_orig:.6f}, CV R2: {r2_orig:.6f}")
            self.oof_preds = final_oof_orig

        final_model = self._build_model(self.best_params)
        final_model.fit(X, y)
        self.models['final'] = final_model
        return self.oof_preds

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.models['final'].predict(X)


class EnsemblePredictor:
    def __init__(self):
        self.meta_learner = None
        self.weights = None
        self.oof_keys: List[str] = []

    def fit(self, X: np.ndarray, y: np.ndarray, oof_dict: dict):
        X_meta = np.column_stack(list(oof_dict.values()))
        logger.info(f"Fitting Ridge meta-learner on {X_meta.shape[1]} base OOF predictions...")
        self.meta_learner = Ridge(alpha=1.0, random_state=cfg.RANDOM_SEED)
        self.meta_learner.fit(X_meta, y)
        self.weights = np.abs(self.meta_learner.coef_) / np.sum(np.abs(self.meta_learner.coef_))
        logger.info(f"Blending weights: {dict(zip(oof_dict.keys(), self.weights))}")
        self.oof_keys = list(oof_dict.keys())

    def predict(self, test_preds_dict: dict) -> np.ndarray:
        X_test_meta = np.column_stack([test_preds_dict[k] for k in self.oof_keys if k in test_preds_dict])

        if X_test_meta.shape[1] == 0:
            logger.warning("No matching predictions found for ensemble. Returning zeros.")
            return np.zeros(X_test_meta.shape[0] if X_test_meta.size > 0 else 1)

        return self.meta_learner.predict(X_test_meta)


def main():
    set_seed(cfg.RANDOM_SEED)
    logger.info("Starting Molecular Property Pipeline...")

    train_path, test_path = DataLoader.find_files('/kaggle/input')
    train_df, test_df = DataLoader.load_data(train_path, test_path)

    desc_gen = DescriptorGenerator(n_jobs=cfg.N_CORES, use_cache=cfg.USE_CACHING)

    X_train_raw = desc_gen.generate(train_df['smiles'], cache_name='train_features')
    X_test_raw = desc_gen.generate(test_df['smiles'], cache_name='test_features')

    common_cols = X_train_raw.columns.intersection(X_test_raw.columns)
    X_train_raw, X_test_raw = X_train_raw[common_cols], X_test_raw[common_cols]
    logger.info(f"Aligned to {len(common_cols)} common features.")

    feat_pipe = FeaturePipeline()
    X_train = feat_pipe.fit_transform(X_train_raw)
    X_test = feat_pipe.transform(X_test_raw)

    del X_train_raw, X_test_raw
    gc.collect()

    all_target_types = train_df['target_type'].unique()
    logger.info(f"Target types present: {all_target_types}")

    final_test_preds = np.zeros(len(test_df))

    if cfg.TARGET_TYPE_STRATEGY == 'separate_models':
        for target_type in all_target_types:
            logger.info(f"========== Processing Target: {target_type} ==========")

            train_mask = (train_df['target_type'] == target_type) & \
                         (train_df['target'].notna()) & \
                         (train_df.index.isin(X_train.index))

            y_subset = train_df.loc[train_mask, 'target'].values
            X_subset = X_train.loc[train_mask]

            if len(y_subset) < 100:
                logger.warning(f"Insufficient training samples ({len(y_subset)}) for {target_type}. Using mean fallback.")
                best_avg_pred = np.mean(y_subset) if len(y_subset) > 0 else 0.0
                test_mask = test_df['target_type'] == target_type
                final_test_preds[test_mask] = best_avg_pred
                continue

            y_subset_log = np.log1p(y_subset)

            test_type_mask = test_df['target_type'] == target_type
            valid_test_indices = test_df.index[test_type_mask & test_df.index.isin(X_test.index)]
            X_test_subset = X_test.loc[valid_test_indices]

            if X_test_subset.shape[0] == 0:
                logger.warning(f"No test samples found for target type {target_type}. Skipping.")
                continue

            train_indices_subset = X_subset.index.tolist()

            if cfg.FEATURE_SELECTION_METHOD == 'importance':
                logger.info("Selecting top features via fast LightGBM importance...")

                finite_mask = np.isfinite(y_subset_log)
                if not np.all(finite_mask):
                    logger.warning(f"Dropping {np.sum(~finite_mask)} invalid target values for {target_type}.")
                    X_subset = X_subset[finite_mask]
                    y_subset_log = y_subset_log[finite_mask]
                    y_subset = y_subset[finite_mask]
                    train_indices_subset = [idx for idx, keep in zip(train_indices_subset, finite_mask) if keep]

                if len(X_subset) == 0:
                    logger.warning(f"No valid samples remaining for {target_type}. Skipping.")
                    continue

                X_subset_np = X_subset.values
                selector_model = lgb.LGBMRegressor(
                    n_estimators=150, num_leaves=63, learning_rate=0.1,
                    random_state=cfg.RANDOM_SEED, n_jobs=cfg.N_CORES, verbose=-1
                )
                selector_model.fit(X_subset_np, y_subset_log)
                importances = selector_model.feature_importances_
                top_idx = np.argsort(importances)[-cfg.N_SELECTED_FEATURES:]
                top_idx = np.sort(top_idx)

                X_subset = X_subset_np[:, top_idx]
                X_test_subset = X_test_subset.values[:, top_idx]
            else:
                X_subset = X_subset.values
                X_test_subset = X_test_subset.values

            model_names = []
            if cfg.ENABLE_LGBM: model_names.append('lightgbm')
            if cfg.ENABLE_XGBOOST: model_names.append('xgboost')
            if cfg.ENABLE_CATBOOST: model_names.append('catboost')
            if cfg.ENABLE_RF: model_names.append('rf')
            if cfg.ENABLE_MLP: model_names.append('mlp')

            if not model_names:
                logger.warning(f"No models enabled for {target_type}. Skipping.")
                continue

            oof_collection = {}
            test_preds_collection = {}

            for m_name in model_names:
                try:
                    trainer = ModelTrainer(m_name)
                    oof_preds_orig = trainer.fit_cv(X_subset, y_subset_log, y_orig=y_subset)
                    oof_collection[m_name] = oof_preds_orig
                    test_preds_collection[m_name] = trainer.predict(X_test_subset)
                    del trainer
                    gc.collect()
                except Exception as e:
                    logger.error(f"Error training {m_name} for {target_type}: {e}")
                    continue

            if not oof_collection:
                logger.warning(f"No models trained successfully for {target_type}. Using mean fallback.")
                final_test_preds[valid_test_indices] = np.mean(y_subset)
                continue

            ensembler = EnsemblePredictor()
            ensembler.fit(X_subset, y_subset, oof_collection)
            final_target_preds = np.maximum(0, ensembler.predict(test_preds_collection))

            final_test_preds[valid_test_indices] = final_target_preds
            logger.info(f"Completed predictions for {target_type} (test samples: {len(final_target_preds)}).")

    else:
        logger.info("Using Multi-Output Strategy (globally trained models).")
        y_train = train_df.loc[train_df.index.isin(X_train.index), 'target'].values
        y_train_log = np.log1p(y_train)

        from sklearn.preprocessing import OneHotEncoder
        enc = OneHotEncoder(sparse_output=False)
        train_types = enc.fit_transform(train_df.loc[X_train.index, ['target_type']])
        test_types = enc.transform(test_df.loc[X_test.index, ['target_type']])
        X_train_global = np.hstack([X_train.values, train_types])
        X_test_global = np.hstack([X_test.values, test_types])

        model_names = []
        if cfg.ENABLE_LGBM: model_names.append('lightgbm')
        if cfg.ENABLE_XGBOOST: model_names.append('xgboost')
        if cfg.ENABLE_CATBOOST and CATBOOST_AVAILABLE: model_names.append('catboost')

        oof_collection = {}
        test_preds_collection = {}

        for m_name in model_names:
            try:
                trainer = ModelTrainer(m_name)
                oof_preds_orig = trainer.fit_cv(X_train_global, y_train_log, y_orig=y_train)
                oof_collection[m_name] = oof_preds_orig
                test_preds_collection[m_name] = trainer.predict(X_test_global)
                del trainer
                gc.collect()
            except Exception as e:
                logger.error(f"Error training {m_name}: {e}")
                continue

        if oof_collection:
            ensembler = EnsemblePredictor()
            ensembler.fit(X_train_global, y_train, oof_collection)
            final_test_preds = np.maximum(0, ensembler.predict(test_preds_collection))
        else:
            logger.error("No models trained successfully. Using baseline prediction.")
            final_test_preds = np.full(len(test_df), np.mean(y_train))

    submission = pd.DataFrame({'id': test_df['id'], 'target': final_test_preds})
    submission.to_csv('submission.csv', index=False)
    logger.info("Submission saved to submission.csv")

    joblib.dump(feat_pipe, 'feature_pipeline.pkl')
    logger.info("Finished! Feature pipeline and models persisted.")


if __name__ == '__main__':
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 52.3 MB/s eta 0:00:00


[2026-08-12 11:27:03] INFO - Starting Molecular Property Pipeline...
[2026-08-12 11:27:03] INFO - Loaded 7409 train samples, 4940 test samples.
[2026-08-12 11:27:03] INFO - Generating descriptors for 7409 molecules (train_features)...
Descriptors[train_features]: 100%|██████████| 7409/7409 [01:21<00:00, 91.01it/s]
[2026-08-12 11:28:25] INFO - Generated 2439 base features for 7409 valid molecules.
[2026-08-12 11:28:26] INFO - Generating descriptors for 4940 molecules (test_features)...
Descriptors[test_features]: 100%|██████████| 4940/4940 [00:56<00:00, 86.77it/s] 
[2026-08-12 11:29:23] INFO - Generated 2439 base features for 4940 valid molecules.
[2026-08-12 11:29:24] INFO - Aligned to 2439 common features.
[2026-08-12 11:29:24] INFO - After VarianceThreshold: 2123 features left.
[2026-08-12 11:29:25] INFO - Target types present: ['tg' 'egc' 'eps' 'eea' 'egb' 'ei' 'nc']
[2026-08-12 11:29:25] INFO - ========== Processing Target: tg ==========
[2026-08-12 11:29:25] INFO - Selecting top f